# 8. Loops, Cycles & Retry Patterns

Graphs, unlike simple chains, can have edges that point **backward**. A cycle isn't a new primitive — it's Video 7's `add_conditional_edges` routing back to a node that already ran, instead of only forward. That's the whole mechanism; this video is about the practical concerns that come *with* it: guaranteeing termination, and using a bounded retry as a realistic example.

**Sections**
1. Recap: the utility + the Screener pipeline (Video 7's router version)
2. A cycle is just a backward conditional edge — no new API
3. Termination: loop counters + `recursion_limit`
4. Applying it: a bounded re-scoring loop on the Applicant Screener
5. Generalizing: retry patterns, and what's still ahead (Video 20)

## 1. Recap, and Priya's next complaint

**Priya's problem:** Harry's score comes out close to the line between "no" and "interview". A borderline score is exactly where the rubric's own noise matters most, and she doesn't want to commit to a decision on a single reading. She wants a **second look**, but only for borderline cases, and not forever.

- **Input:** a resume (we run all three candidates).
- **Output:** the same `decision` and `summary` as Video 7, plus a `rescore_attempts` count, and notes that show every scoring pass.
- **Expected:** Hermione and Ron never enter the loop. Harry does, once, and comes out as an `interview`.

Same `llm_utils.py`, same Screener and router-based three-way branch from Video 7. If you're running this notebook fresh, the cells below recreate everything up to (not including) today's loop.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

for env_var in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY"]:
    if not os.environ.get(env_var):
        os.environ[env_var] = getpass.getpass(f"{env_var}: ")

In [ ]:
%%writefile llm_utils.py
"""Provider-agnostic LLM utility used by every node in this course from Video 3 onward.

Call `call_llm(provider, messages, system=None)` instead of ever importing a provider
SDK directly from graph/node code. Swapping models is then a one-line `provider` change.
"""
from dataclasses import dataclass

import anthropic
from openai import OpenAI
from google import genai
from google.genai import types as genai_types

# Standardize on one current, non-legacy model per provider.
CLAUDE_MODEL = "claude-sonnet-5"
GPT_MODEL = "gpt-5.1"
GEMINI_MODEL = "gemini-3.5-flash"

DEFAULT_PROVIDER = "claude"  # the rest of this course leaves this default in place

_anthropic_client = anthropic.Anthropic()
_openai_client = OpenAI()
_gemini_client = genai.Client()


@dataclass
class LLMResponse:
    content: str
    provider: str
    model: str
    raw: object  # the untouched provider response, for debugging


def _call_claude(messages: list[dict], system: str | None) -> LLMResponse:
    resp = _anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system=system or anthropic.NOT_GIVEN,
        messages=messages,
    )
    # Newer Claude models may put a thinking block first, so join only the text blocks.
    text = "".join(block.text for block in resp.content if block.type == "text")
    return LLMResponse(content=text, provider="claude", model=CLAUDE_MODEL, raw=resp)


def _call_gpt(messages: list[dict], system: str | None) -> LLMResponse:
    full_messages = ([{"role": "system", "content": system}] if system else []) + messages
    resp = _openai_client.chat.completions.create(model=GPT_MODEL, messages=full_messages)
    return LLMResponse(content=resp.choices[0].message.content, provider="gpt", model=GPT_MODEL, raw=resp)


def _to_gemini_contents(messages: list[dict]) -> list[genai_types.Content]:
    role_map = {"user": "user", "assistant": "model"}
    return [
        genai_types.Content(role=role_map[m["role"]], parts=[genai_types.Part(text=m["content"])])
        for m in messages
    ]


def _call_gemini(messages: list[dict], system: str | None) -> LLMResponse:
    config = genai_types.GenerateContentConfig(system_instruction=system) if system else None
    resp = _gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=_to_gemini_contents(messages),
        config=config,
    )
    return LLMResponse(content=resp.text, provider="gemini", model=GEMINI_MODEL, raw=resp)


_PROVIDERS = {"claude": _call_claude, "gpt": _call_gpt, "gemini": _call_gemini}


def call_llm(provider: str = DEFAULT_PROVIDER, messages: list[dict] | None = None, system: str | None = None) -> LLMResponse:
    if provider not in _PROVIDERS:
        raise ValueError(f"Unknown provider {provider!r}, expected one of {list(_PROVIDERS)}")
    return _PROVIDERS[provider](messages or [], system)


In [ ]:
import operator
from typing import Annotated, TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

from llm_utils import call_llm

# The three candidates we follow through every notebook in this story.
RESUMES = {
    "Hermione": """
Hermione Granger
7 years of experience as a senior backend software engineer.
Skills: Python, distributed systems, SQL, PostgreSQL, Kubernetes, API design.
Led a 6-person platform team at a fintech company, mentoring junior engineers.
""",
    "Harry": """
Harry Potter
4 years of experience as a software developer.
Skills: Python, SQL, REST APIs.
Informally mentored two interns. No distributed systems or Kubernetes experience.
""",
    "Ron": """
Ron Weasley
8 months of experience as a junior web developer.
Skills: HTML, CSS, basic JavaScript.
No team or leadership experience.
""",
}

PARSE_SYSTEM_PROMPT = (
    "You extract structured fields from a resume. Reply with exactly four lines, "
    "no extra commentary:\n"
    "Name: <candidate name>\n"
    "Years: <years of experience as a number>\n"
    "Skills: <comma-separated skills>\n"
    "Leadership: <one short phrase on leadership or mentoring experience, or none>"
)

RUBRIC = (
    "Score the candidate from 0-100 against this fixed rubric:\n"
    "- 5+ years experience: up to 40 points\n"
    "- Skills overlap with [Python, distributed systems, Kubernetes, SQL]: up to 40 points\n"
    "- Leadership experience mentioned: up to 20 points\n"
    "Reply with exactly two lines, no extra commentary:\n"
    "Score: <integer 0-100>\n"
    "Rationale: <one sentence>"
)


def show_graph(graph):
    """Render a compiled graph. Falls back to Mermaid text if the PNG renderer is unavailable."""
    try:
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(graph.get_graph().draw_mermaid())


def parse_resume(state: dict) -> dict:
    result = call_llm("claude", messages=[{"role": "user", "content": state["resume_text"]}], system=PARSE_SYSTEM_PROMPT)
    fields = {"candidate_name": "", "years_experience": "", "skills": "", "leadership": ""}
    for line in result.content.strip().splitlines():
        if line.startswith("Name:"):
            fields["candidate_name"] = line.removeprefix("Name:").strip()
        elif line.startswith("Years:"):
            fields["years_experience"] = line.removeprefix("Years:").strip()
        elif line.startswith("Skills:"):
            fields["skills"] = line.removeprefix("Skills:").strip()
        elif line.startswith("Leadership:"):
            fields["leadership"] = line.removeprefix("Leadership:").strip()
    fields["evaluation_notes"] = [f"Parsed fields for {fields['candidate_name'] or 'candidate'}."]
    return fields


def score_against_rubric(state: dict) -> dict:
    candidate_summary = (
        f"Name: {state['candidate_name']}\n"
        f"Years of experience: {state['years_experience']}\n"
        f"Skills: {state['skills']}\n"
        f"Leadership: {state['leadership']}"
    )
    result = call_llm("claude", messages=[{"role": "user", "content": candidate_summary}], system=RUBRIC)
    score, rationale = 0, ""
    for line in result.content.strip().splitlines():
        if line.startswith("Score:"):
            score = int("".join(ch for ch in line if ch.isdigit()) or 0)
        elif line.startswith("Rationale:"):
            rationale = line.removeprefix("Rationale:").strip()
    return {
        "score": score,
        "score_rationale": rationale,
        "evaluation_notes": [f"Scored {score}/100: {rationale}"],
    }


def reject_node(state: dict) -> dict:
    return {"decision": f"Not a match ({state['score']}/100): {state['score_rationale']}"}


def interview_node(state: dict) -> dict:
    return {"decision": f"Move to interview ({state['score']}/100): {state['score_rationale']}"}


def strong_match_node(state: dict) -> dict:
    return {"decision": f"Strong match ({state['score']}/100): {state['score_rationale']}"}


def generate_summary(state: dict) -> dict:
    prompt = (
        f"Candidate: {state['candidate_name']}\n"
        f"Decision: {state['decision']}\n\n"
        "Write a two-sentence plain-text note for the hiring manager explaining this decision and the suggested next step."
    )
    result = call_llm("claude", messages=[{"role": "user", "content": prompt}])
    return {"summary": result.content}

## 2. A cycle is just a backward conditional edge — no new API

Before touching the Screener, the mechanism in miniature: a router that sends the graph back to a node it already visited, using the exact same `add_conditional_edges` from Video 7. Nothing about the edge API changes — only which direction it happens to point.

In [ ]:
class CountState(TypedDict):
    count: int


def increment(state: CountState) -> dict:
    return {"count": state["count"] + 1}


def keep_going(state: CountState) -> str:
    return "increment" if state["count"] < 3 else "done"


cycle_builder = StateGraph(CountState)
cycle_builder.add_node("increment", increment)
cycle_builder.add_node("done", lambda state: {})
cycle_builder.add_edge(START, "increment")
cycle_builder.add_conditional_edges("increment", keep_going, {"increment": "increment", "done": "done"})
cycle_builder.add_edge("done", END)

cycle_graph = cycle_builder.compile()
show_graph(cycle_graph)
cycle_graph.invoke({"count": 0})

`increment` routes back to itself three times before `keep_going` finally returns `"done"`. That backward arrow in the diagram above is the entire mechanism — everything else in this video is about making sure it terminates.

## 3. Termination: loop counters + `recursion_limit`

Two things are required to guarantee a cycle ends:
1. **State the router can actually see changing** — `keep_going` above only works because `count` moves toward its exit condition on every pass. A routing condition that never changes is an infinite loop.
2. **`recursion_limit`** as the safety net LangGraph provides regardless — a hard ceiling on total graph steps, raising `GraphRecursionError` once hit, so a bug in your termination condition doesn't just hang forever.

Here's what happens when the termination condition is broken — deliberately, to see the safety net catch it:

In [ ]:
from langgraph.errors import GraphRecursionError


def broken_keep_going(state: CountState) -> str:
    return "increment"  # bug: never returns "done", regardless of count


broken_builder = StateGraph(CountState)
broken_builder.add_node("increment", increment)
broken_builder.add_node("done", lambda state: {})
broken_builder.add_edge(START, "increment")
broken_builder.add_conditional_edges("increment", broken_keep_going, {"increment": "increment", "done": "done"})
broken_builder.add_edge("done", END)
broken_graph = broken_builder.compile()

try:
    broken_graph.invoke({"count": 0}, {"recursion_limit": 5})
except GraphRecursionError as e:
    print(f"Caught it: {e}")

`recursion_limit` is a safety net for bugs, not a substitute for a real exit condition — the goal is always to design the loop so it terminates on its own well before the limit, as the Screener's retry loop does next.

## 4. Applying it: a bounded re-scoring loop on the Applicant Screener

When `score_against_rubric` produces a **borderline** result (middle band, 50–64), route back to an `adjust_and_rescore` node that applies a relaxed rubric instead of immediately forcing a reject/interview decision. Cap it at a small fixed number of attempts via a counter field, then fall through to Video 7's three-way branching regardless of outcome. The counter uses `operator.add`: this is exactly the kind of field Video 6 warned about, touched on every loop pass, so it needs accumulation, not overwrite.

One design rule keeps the story honest: a second look can lift a borderline candidate to **`interview`**, never straight to `strong_match`, so the relaxed rubric caps its score below the strong-match line. (In a real hiring system, loosening a rubric for people near the line deserves scrutiny. It's here as a teaching device for loops, not as screening advice.)

In [ ]:
MAX_RESCORE_ATTEMPTS = 2

RELAXED_RUBRIC = (
    "Re-score the candidate from 0-100 against this RELAXED rubric - give the benefit of the "
    "doubt on borderline cases:\n"
    "- 3+ years experience: up to 40 points\n"
    "- Any overlap with [Python, distributed systems, Kubernetes, SQL, backend, API]: up to 40 points\n"
    "- Any hint of leadership, mentorship, or team responsibility: up to 20 points\n"
    "A second look can lift a borderline candidate to the interview stage but never to a strong "
    "match, so the final score must not exceed 79. Do not mention this cap in your rationale.\n"
    "Reply with exactly two lines, no extra commentary:\n"
    "Score: <integer 0-79>\n"
    "Rationale: <one sentence>"
)


class ScreenerState(TypedDict):
    resume_text: str
    candidate_name: str
    years_experience: str
    skills: str
    leadership: str
    score: int
    score_rationale: str
    decision: str
    summary: str
    rescore_attempts: Annotated[int, operator.add]
    evaluation_notes: Annotated[list[str], operator.add]


def adjust_and_rescore(state: ScreenerState) -> dict:
    candidate_summary = (
        f"Name: {state['candidate_name']}\n"
        f"Years of experience: {state['years_experience']}\n"
        f"Skills: {state['skills']}\n"
        f"Leadership: {state['leadership']}"
    )
    result = call_llm("claude", messages=[{"role": "user", "content": candidate_summary}], system=RELAXED_RUBRIC)
    score, rationale = state["score"], state["score_rationale"]
    for line in result.content.strip().splitlines():
        if line.startswith("Score:"):
            score = int("".join(ch for ch in line if ch.isdigit()) or 0)
        elif line.startswith("Rationale:"):
            rationale = line.removeprefix("Rationale:").strip()
    attempt_number = state["rescore_attempts"] + 1
    return {
        "score": score,
        "score_rationale": rationale,
        "rescore_attempts": 1,  # delta - operator.add accumulates this across passes
        "evaluation_notes": [f"Rescore attempt {attempt_number}: {score}/100 ({rationale})"],
    }


def route_after_scoring(state: ScreenerState) -> str:
    if 50 <= state["score"] < 65 and state["rescore_attempts"] < MAX_RESCORE_ATTEMPTS:
        return "rescore"
    elif state["score"] < 50:
        return "reject"
    elif state["score"] < 80:
        return "interview"
    else:
        return "strong_match"


loop_builder = StateGraph(ScreenerState)
loop_builder.add_node("parse_resume", parse_resume)
loop_builder.add_node("score_against_rubric", score_against_rubric)
loop_builder.add_node("adjust_and_rescore", adjust_and_rescore)
loop_builder.add_node("reject", reject_node)
loop_builder.add_node("interview", interview_node)
loop_builder.add_node("strong_match", strong_match_node)
loop_builder.add_node("generate_summary", generate_summary)

loop_builder.add_edge(START, "parse_resume")
loop_builder.add_edge("parse_resume", "score_against_rubric")

branch_map = {"rescore": "adjust_and_rescore", "reject": "reject", "interview": "interview", "strong_match": "strong_match"}
loop_builder.add_conditional_edges("score_against_rubric", route_after_scoring, branch_map)
loop_builder.add_conditional_edges("adjust_and_rescore", route_after_scoring, branch_map)  # same router - this edge is the backward arrow

loop_builder.add_edge("reject", "generate_summary")
loop_builder.add_edge("interview", "generate_summary")
loop_builder.add_edge("strong_match", "generate_summary")
loop_builder.add_edge("generate_summary", END)

screener_loop_graph = loop_builder.compile()
show_graph(screener_loop_graph)

In [ ]:
for name, resume in RESUMES.items():
    print(f"=== {name} ===")
    result = screener_loop_graph.invoke({"resume_text": resume, "rescore_attempts": 0, "evaluation_notes": []})
    print(f"final score: {result['score']}  |  attempts used: {result['rescore_attempts']}")
    print(f"decision: {result['decision']}")
    print(f"summary: {result['summary']}")
    for note in result["evaluation_notes"]:
        print("-", note)
    print()

Note `score_against_rubric` and `adjust_and_rescore` both route through the *same* `route_after_scoring` function and the *same* mapping. The only thing that makes this a cycle rather than two separate acyclic edges is that `"rescore"` happens to map back to a node the graph already visited.

Look at the three runs: **Hermione** and **Ron** never land in the borderline band, so for them this loop never runs and the graph behaves exactly like Video 7's version (`rescore_attempts` stays at 0). **Harry** is scored once against the strict rubric, falls in the 50–64 band, goes through `adjust_and_rescore`, and comes out as an `interview`. His notes show both passes. It's capped at `MAX_RESCORE_ATTEMPTS` passes before falling through regardless. Try lowering `MAX_RESCORE_ATTEMPTS` to `0` to confirm the branch logic still falls through correctly with no rescoring at all.

## 5. Generalizing: retry patterns, and what's still ahead

The same shape — **try, check a condition, loop back on failure, cap the attempts** — is also how you'd retry a failed LLM call or a failing tool call, not just re-evaluate a borderline score. The graph-cycle mechanism is identical either way.

What this video does *not* cover: this is a hand-rolled, logical retry built from a counter and a routing condition — it says nothing about *infrastructure*-level reliability (crash recovery, surviving a process restart mid-loop, exponential backoff). That's Video 20's job, once checkpointing (Video 14) exists to build it on. Today's takeaway is narrower and more fundamental: **a cycle is just a conditional edge that points backward**, and termination is something *you* design for, with `recursion_limit` as the backstop when you get it wrong.

**Next video** covers the last core mechanic before tools and agents: parallel fan-out/fan-in. Priya now wants a fuller picture of each candidate (skills, experience and leadership judged separately), and she wants it fast. That finally gives `evaluation_notes` its real payoff: multiple nodes writing to the same reducer-backed field in the *same* step, not just across sequential loop passes.